In [ ]:
import os
import sys
import requests
import shutil
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
import kagglehub


print("Installing dependencies and cloning repositories...")
!pip install -q timm yacs iopath kagglehub
!pip install -q git+https://github.com/fra31/auto-attack.git

os.chdir('/content')
if os.path.exists('/content/SAR'): shutil.rmtree('/content/SAR')
if os.path.exists('/content/test-time-adaptation'): shutil.rmtree('/content/test-time-adaptation')

!git clone https://github.com/mr-eggplant/SAR.git /content/SAR
!git clone --depth 1 https://github.com/mariodoebler/test-time-adaptation.git /content/test-time-adaptation


sys.path.append('/content/SAR')
sys.path.insert(0, '/content/test-time-adaptation/classification')


import sar
def stable_softmax_entropy(x):
    return -(x.softmax(1) * x.log_softmax(1)).sum(1)
sar.softmax_entropy = stable_softmax_entropy


import types
import robustbench
try:
    import robustbench.loaders as rb_loaders
except ImportError:
    rb_loaders = types.ModuleType('loaders')
    robustbench.loaders = rb_loaders
    sys.modules['robustbench.loaders'] = rb_loaders

class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, *args, **kwargs): pass
    def __len__(self): return 0
rb_loaders.CustomCifarDataset = DummyDataset
rb_loaders.CustomImageFolder = datasets.ImageFolder


# Attack on ViT + ROID

In [ ]:
import os
import requests
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


print("Initializing TSMA Surrogate...")
attack_model_resnet = timm.create_model('resnet50', pretrained=True).to(device).eval()
for p in attack_model_resnet.parameters():
    p.requires_grad = False

def tsma_attack(images, labels, epsilon=24/255, alpha=4/255, steps=3, lambda_drift=0.05):
    mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(images.device)
    std  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(images.device)

    images_unnorm = images * std + mean

    delta = torch.zeros_like(images_unnorm).uniform_(-epsilon, epsilon)
    delta = torch.clamp(images_unnorm + delta, 0, 1) - images_unnorm
    delta.requires_grad_(True)
    momentum = torch.zeros_like(delta)
    
    with torch.no_grad():
        clean_feat = attack_model_resnet.forward_features(images)
        clean_mu = clean_feat.mean(dim=(2,3))
        clean_var = clean_feat.var(dim=(2,3))

    for _ in range(steps):
        adv = (images_unnorm + delta - mean) / std

        outputs = attack_model_resnet(adv)
        
        adv_feat = attack_model_resnet.forward_features(adv)
        adv_mu = adv_feat.mean(dim=(2,3))
        adv_var = adv_feat.var(dim=(2,3))
        
        probs = F.softmax(outputs, dim=1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1).mean()
        

        loss_ce = F.cross_entropy(outputs, labels)
        
        probs = F.softmax(outputs, dim=1)
        loss_entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1).mean()
        
        loss_drift = F.mse_loss(adv_mu, clean_mu) + F.mse_loss(adv_var, clean_var)
        
        loss = -loss_ce + (2.0 * loss_entropy) - (lambda_drift * loss_drift)
        if delta.grad is not None:
            delta.grad.zero_()

        loss.backward()

        with torch.no_grad():
            
            momentum = 0.9 * momentum + delta.grad.sign()
            delta += alpha * momentum.sign()
            delta.clamp_(-epsilon, epsilon)
            delta.copy_(torch.clamp(images_unnorm + delta, 0, 1) - images_unnorm)

    adv = images_unnorm + delta.detach()
    return (adv - mean) / std


class ROID(nn.Module):
    def __init__(self, model, lr=2.5e-4, alpha=0.99, beta=0.9, tau=1/3):
        super().__init__()
        self.model = model
        self.model_0 = copy.deepcopy(model).eval()
        self.model_0.requires_grad_(False)

        params = []
        for m in self.model.modules():
            if isinstance(m, nn.LayerNorm):
                m.requires_grad_(True)
                params += [m.weight, m.bias]
            else:
                m.requires_grad_(False)

        self.optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9)
        self.alpha, self.beta, self.tau = alpha, beta, tau
        self.tendency = None

    def adapt(self, x):
        self.model.train()
        x_aug = transforms.functional.hflip(x)

        out = self.model(x)
        out_aug = self.model(x_aug)

        p = out.softmax(1)
        p_aug = out_aug.softmax(1)

        mean_p = p.mean(0).detach()
        self.tendency = mean_p if self.tendency is None else self.beta*self.tendency + (1-self.beta)*mean_p

        sim = F.cosine_similarity(p.detach(), self.tendency.unsqueeze(0), dim=1)
        w = torch.exp((1 - sim) / self.tau)

        loss = (w * (-(p * torch.log(p+1e-6)).sum(1))).mean() + \
               (-(p * torch.log(p_aug+1e-6)).sum(1)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        with torch.no_grad():
            for p1, p0 in zip(self.model.parameters(), self.model_0.parameters()):
                if p1.requires_grad:
                    p1.data = self.alpha*p1.data + (1-self.alpha)*p0.data

        self.model.eval()
        with torch.no_grad():
            return self.model(x).softmax(1)


print("Loading ViT...")
model = timm.create_model('vit_base_patch16_224', pretrained=True).to(device)

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

dataset = datasets.ImageFolder(data_path, transform=transform)


url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}

dataset.samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
                   for p,_ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.targets = [s[1] for s in dataset.samples]

loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)


roid = ROID(model).to(device)

correct_total, total_total = 0, 0
correct_clean, total_clean = 0, 0
correct_pois, total_pois = 0, 0

POISON_FREQ = 2

pbar = tqdm(enumerate(loader), total=len(loader), desc="ViT + ROID + Attack")

for i, (images, labels) in pbar:
    images, labels = images.to(device), labels.to(device)

    is_poisoned = (i % POISON_FREQ == 0)

    if is_poisoned:
        inputs = tsma_attack(images, labels)
    else:
        inputs = images

    outputs = roid.adapt(inputs)
    preds = outputs.argmax(1)

    correct = (preds == labels).sum().item()
    total = labels.size(0)

    correct_total += correct
    total_total += total

    if is_poisoned:
        correct_pois += correct
        total_pois += total
    else:
        correct_clean += correct
        total_clean += total

    pbar.set_postfix({
        "Total": f"{100*correct_total/total_total:.2f}",
        "Clean": f"{100*correct_clean/max(1,total_clean):.2f}",
        "Pois": f"{100*correct_pois/max(1,total_pois):.2f}",
        "Pois?": "Y" if is_poisoned else "N"
    })

print("\nFINAL RESULTS")
print(f"Total: {100*correct_total/total_total:.2f}%")
print(f"Clean: {100*correct_clean/max(1,total_clean):.2f}%")
print(f"Pois : {100*correct_pois/max(1,total_pois):.2f}%")